This project is a work in progress. 
Features to add: 
-predict how a misspelled word would be pronounced using the phonetic rules of English
-better predict next word using POS tagging

This first cell contains the necessary libraries for this Jupyter notebook, as well as configuration variables to fine-tune the functions in the sentence prediction/spellchecking model.

In [ ]:
import nltk
import re
import math
import string

from nltk import pos_tag # possible expansion to better predict which type of word is next
from nltk.corpus import cmudict, brown, reuters, stopwords
from collections import defaultdict, Counter

downloads = ["cmudict", "brown", "reuters", 'averaged_perceptron_tagger', 
             'averaged_perceptron_tagger_eng', 'punkt']

for download in downloads:
    nltk.download(download, force=True)

# ---------- configuration ----------
LAMBDA = [0.05, 0.15, 0.2, 0.6] # format: [unigram_weight, bigram_weight, trigram_weight, 4-gram_weight]; sum(LAMBDA) ≈ 1.0; longer n-grams have more weight
MAX_NGRAM = len(LAMBDA)
SMOOTH = 1e-8 # generally a fallback value in case an n-gram has a count of 0

[nltk_data] Downloading package cmudict to
[nltk_data]     C:\Users\uddin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\cmudict.zip.
[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\uddin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\brown.zip.
[nltk_data] Downloading package reuters to
[nltk_data]     C:\Users\uddin\AppData\Roaming\nltk_data...
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\uddin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\uddin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\uddin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping taggers\averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\uddin\A

This second cell contains all of the functions that precompute variables that are used in the model -- such as the full corpus, n-gram counts, sum of the number of each n-gram.

In [7]:
def fuse_corpora(corpus_one, corpus_two, limit=None):
    """
    Combine two corpora (that have .words() method) into a single list of lower-cased words.
    """
    return [word.lower() for word in corpus_one.words()[:limit]] + [word.lower() for word in corpus_two.words()[:limit]]

def smooth_value():
    """
    Returns a small value to be used for smoothing n-gram counts.
    """
    return SMOOTH

def build_counts_model(corpus, min_size=1, max_size=MAX_NGRAM, smooth=SMOOTH):
    """
    Returns the number of occurences of each ngram in a corpus, sorted by the ngram size.
    """
    counts = {}
    L = len(corpus)

    for n in range(min_size, max_size + 1):
        counter = Counter(
            tuple(corpus[i:i+n]) for i in range(L - n + 1)
        )

        d = defaultdict(smooth_value)
        d.update(counter)
        counts[n] = d

    return counts

def precompute_prefix_denominators(counts: dict, max_size: int = MAX_NGRAM):
    """
    Build a lookup: prefix_sums[n][prefix] = total count of n-grams starting with that prefix.
    """
    prefix_sums = {n: defaultdict(int) for n in range(2, max_size + 1)}

    for n in range(2, max_size + 1):
        counter = counts.get(n)

        if not counter:
            continue

        ps = prefix_sums[n]
        
        for ngram, freq in counter.items():
            ps[ngram[:-1]] += freq

    return prefix_sums

def precompute_totals(counts: dict, smooth: float = SMOOTH):
    """
    Precompute total smoothed counts for each n-gram order.
    """
    total_counts = {}

    for n, counter in counts.items():
        counter_len = len(counter)
        counter_sum = sum(counter.values())
        total_counts[n] = counter_sum + smooth * counter_len

    return total_counts

This third cell computes all the variables that will be used in the final model. This code can take at least 30 seconds to execute, so print statements are included to indicate if the cell is running properly.

In [ ]:
# build corpus
corpus = fuse_corpora(brown, reuters)
print("corpus built")

# counts of unigrams to 4-grams
counts = build_counts_model(corpus)
print("counts models built")

# precompute prefix denominator counts
prefix_denominator = precompute_prefix_denominators(counts)
print("full prefix denominators precomputed")

# precompute total counts
total_counts = precompute_totals(counts)
print("total counts precomputed")

# create pronunciation dictionary using cmudict
pronounce_dictionary = cmudict.dict()
print(f"pronunciation of the word cat: {pronounce_dictionary['cat']}") # outputs [['K', 'AE1', 'T']]

# create list of valid keys in pronounceDict
valid_words = list(pronounce_dictionary.keys())

# create dictionary that maps pronunciations tuples to words
reverse_pronounce_dictionary = defaultdict(list)

for word, pronunciations in pronounce_dictionary.items():

    for pronunciation in pronunciations:
        
        reverse_pronounce_dictionary[tuple(pronunciation)].append(word)

print(f"words that are pronounced KAE1T: {reverse_pronounce_dictionary[('K', 'AE1', 'T')]}") # outputs ['cat', 'catt', 'kat', 'katt']

# Collect words tagged as adjectives, prepositions, and nouns in Brown to iron out common their/they're/there errors
adjectives = set()
prepositions = set()
nouns = set()

for (word, tag) in brown.tagged_words():
    
    word_lc = word.lower()

    if tag.startswith("JJ"):
        adjectives.add(word_lc)

    if tag.startswith("IN"):
        prepositions.add(word_lc)

    if tag.startswith("NN"):
        nouns.add(word_lc)

nouns.remove("a") # fixes a tagging error

corpus built
full counts models built
full prefix denominators precomputed
total counts precomputed
pronunciation of the word cat: [['K', 'AE1', 'T']]
words that are pronounced KAE1T: ['cat', 'catt', 'kat', 'katt']


The following cells contain the functions are used in the model.

In [ ]:
def prob_from_counts(word: str, context_size: int, context_data: tuple, 
                     counts: dict, prefix_sums: dict, total_counts: dict, SMOOTH=1e-6):
    """
    Return P(word | context) estimated from counts and precomputed denominators.
    """
    n = context_size + 1
    counter = counts.get(n)
    if counter is None:
        counter = Counter()
    total = total_counts.get(n)
    if total is None:
        total = sum(counter.values()) + SMOOTH * len(counter)
    
    if context_size == 0:
        # unigram probability
        return (counter.get((word,), 0) + SMOOTH) / total

    # Compute prefix
    if context_size <= len(context_data):
        prefix = tuple(context_data[-context_size:])
    else:
        return SMOOTH / (total + SMOOTH)

    # Get denominator for the conditional probability
    denom = prefix_sums.get(n, {}).get(prefix, 0) + SMOOTH * len(counter)
    joint_count = counter.get(prefix + (word,), 0) + SMOOTH

    # Safe fallback
    if denom == 0:
        return SMOOTH / (total + SMOOTH)

    return joint_count / denom



# Tests
print(prob_from_counts("year", 2, ("in", "the", "last"), counts, prefix_denominator, total_counts))
print(prob_from_counts("day", 2, ("in", "the", "last"), counts, prefix_denominator, total_counts))

0.03823796095285799
0.02761630572271161


In [10]:
def nbest_continuations(sent: list, counts: dict, prefix_sums: int, total_counts: int, nbest: int=5):
    """
    Outputs the most likely next word of the input sentence.
    """
    ctx = [w.lower() for w in sent]
    candidate_probs = defaultdict(float)

    # Gather candidates from higher-order contexts first
    candidates = set()

    for ngram_size in range(2, min(len(ctx) + 1, MAX_NGRAM + 1)):
        prefix = tuple(ctx[-(ngram_size - 1):])
        
        # collect all words that appear after this prefix
        continuations = [
            ngram[-1]
            for ngram, count in counts[ngram_size].items()
            if ngram[:-1] == prefix
        ]
        
        top_continuations = sorted(
            continuations,
            key=lambda w: counts[ngram_size][prefix + (w,)],
            reverse=True
        )[:100]
        candidates.update(top_continuations)
    
    # fallback to top unigrams only if no higher-order candidates were found
    if len(candidates) == 0:
        top_unigrams = sorted(counts[1], key=counts[1].get, reverse=True)[:300]
        candidates.update(top_unigrams)

    # Compute interpolated log probabilities
    for w in candidates:
        p_mix = 0.0
        for context_size in range(min(len(ctx),MAX_NGRAM)):
            p_mix += LAMBDA[context_size] * prob_from_counts(
                w, context_size, ctx, counts, prefix_sums, total_counts
            )
        candidate_probs[w] = math.log(p_mix)

    ranked = sorted(candidate_probs.items(), key=lambda x: x[1], reverse=True)
    return ranked[:nbest]

In [ ]:
def close_words(word, pronounce_dict=pronounce_dictionary, reverse_pronounce_dict=reverse_pronounce_dictionary):
    """
    Return all words with the same pronunciations.
    """
    if word not in pronounce_dict:
        return []
        # run phonetic spellchecker
        # 1. predict pronunciation of the word using linguistic onset, nucleus, and coda rules
        # 2. find all words with the same predicted pronunciation
        # 3. return as a list of 1-tuples

    results = set()
    for pronunciation in pronounce_dict[word]:
        pronunciation = tuple(pronunciation)
        
        if pronunciation in reverse_pronounce_dict:
            results.update(reverse_pronounce_dict[pronunciation])

    return list(results)

def predict_pronunciation(word):
    letter_pronunciations = {
        "a": ["AH0", "AE1", "EY1"],
        "b": ["B"],
        "c": ["K", "S"],
        "d": ["D"],
        "e": ["EH1", "IY1"],
        "f": ["F"],
        "g": ["G", "JH"],
        "h": ["HH"],
        "i": ["IH1", "AY1"],
        "j": ["JH"],
        "k": ["K"],
        "l": ["L"],
        "m": ["M"],
        "n": ["N"],
        "o": ["OW1", "AO1"],
        "p": ["P"],
        "q": ["K", "KW"],
        "r": ["R"],
        "s": ["S", "Z"],
        "t": ["T"],
        "u": ["UH1", "YUW1"],
        "v": ["V"],
        "w": ["W"],
        "x": ["K", "S"],
        "y": ["Y", "AY1"],
        "z": ["Z", "ZH"],

        "th": ["TH", "DH"],
        "ng": ["NG"],
        "sh": ["SH"],
        "ch": ["CH"],
    }

    sonority_ranks = {
        "vowel": 1, 
        "approximant": 2, 
        "nasal": 3, 
        "fricative": 4, 
        "affricative": 5, 
        "stop": 6
    }

    letter_classification = {
        "a": "vowel", "e": "vowel", "i": "vowel", 
        "o": "vowel", "u": "vowel",

        "l": "approximant", "r": "approximant",
        "w": "approximant", "y": "approximant",

        "m": "nasal", "n": "nasal", "ng": "nasal",

        "f": "fricative", "v": "fricative",
        "s": "fricative", "z": "fricative",
        "h": "fricative",

        "ch": "affricative", "j": "affricative",

        "b": "stop", "d": "stop", "g": "stop",
        "k": "stop", "p": "stop", "t": "stop"
    }

    nucleus_present = False
    previous_sonority_rank = 1
    word_lower = word.lower()
    results = [""]

    # iterate through each letter in the word and determine its classification, possible pronunciations, and syllable structure
    for c, i in enumerate(word_lower):
        current_letter_class = letter_classification[c]
        pronunciation_variation = 0

        # switch to nucelus/onset determination mode
        if current_letter_class == "vowel":
            nucleus_present = True

        # create a pseudo-tree data structure of all possible pronunciations of the word
        variation_count = len(letter_pronunciations[c])
        results *= variation_count

        # determine if the current letter is a coda consonant, and if latter, check relative sonority 
        # to the previous letter to determine if it is part of the same syllable or a new syllable
        if nucleus_present and current_letter_class != "vowel":
            previous_sonority_rank = 1

            # check if current letter is less sonorous than previous letter; if so, 
            # it is part of the same syllable, otherwise it is a new syllable
            if sonority_ranks[current_letter_class] > previous_sonority_rank:

                # add all possible pronunciations of the current letter to the current syllable
                for result in results:
                    result += letter_pronunciations[c][pronunciation_variation]
                    pronunciation_variation = (pronunciation_variation + 1) % variation_count

                # update the previous sonority rank to the current letter's sonority rank
                previous_sonority_rank = sonority_ranks[current_letter_class]
            
            # add a hyphen to the current syllable and add all possible 
            # pronunciations of the current letter to the new syllable
            else:
                for result in results:
                    result += "-" + letter_pronunciations[c][pronunciation_variation]
                    pronunciation_variation = (pronunciation_variation + 1) % variation_count

                # exit coda determination mode
                nucleus_present = False
        
        # add onset/nucelus consonant to the current syllable
        else: 
            for result in results:
                result += letter_pronunciations[c][pronunciation_variation]
                pronunciation_variation = (pronunciation_variation + 1) % variation_count



NameError: name 'pronounce_dictionary' is not defined

In [ ]:
punctuation = set(string.punctuation)

def nextLastWordIndex(sent, current_index, punct=punctuation):
    """
    Finds the index of the word before the next punctuation mark in a sentence.
    """
    for i in range(current_index + 1, len(sent)):
        if sent[i] in punct:
            return i - 1
        
    return len(sent) - 1  # return last index of sentence if no punctuation found

def best_word(sent: list, wordIndex, candidates,
             counts_full, prefix_denoms_full, total_counts_full,
             smooth=SMOOTH, lambda_vals=LAMBDA, maxSize=MAX_NGRAM, adjs=adjectives, ins=prepositions, nns=nouns):
    """
    Chooses the best candidate replacement for sent[wordIndex] using n-gram context.
    Uses both past and future context. Returns the candidate with highest log-prob.
    """

    wordProbs = defaultdict(float)
    stopIndex = nextLastWordIndex(sent, wordIndex)  # punctuation boundary

    for cand in candidates:
        total_logp = 0.0

        # ----- 1. PAST CONTEXT -----
        # compute P(cand | previous words)
        for n in range(len(lambda_vals)):
            total_logp += lambda_vals[n-1] * prob_from_counts(
                cand,
                n,
                sent[:wordIndex] + [cand],
                counts_full, prefix_denoms_full, total_counts_full
            )

        # ----- 2. FUTURE CONTEXT -----
        # compute P(next_word | cand), P(next2 | cand next), ...
        future_slice = sent[wordIndex+1: stopIndex]
        context_window = [cand] + future_slice  # synthetic window

        for i in range(len(context_window) - 1):
            next_token = context_window[i + 1]

            # n-gram size grows as we move right
            n = min(maxSize, i + 2)

            total_logp += prob_from_counts(
                next_token,
                n,
                context_window[:i+1],
                counts_full, prefix_denoms_full, total_counts_full,
            )

            if n == maxSize:
                break  # no larger n-grams

        wordProbs[cand] = total_logp

    wordProbs[sent[wordIndex]] += smooth   # small boost for original word

    # cases for THEY'RE / THEIR / THERE
    if sent[wordIndex] in {"they're", 'their', 'there'} and wordIndex < len(sent) - 1:
        for i in range(min(len(sent) - wordIndex, maxSize)):
            if sent[wordIndex + i] in ins:
                wordProbs["they're"] += 1
                break

            if sent[wordIndex + i] in adjs:
                continue

    # pick the highest-scoring candidate
    return max(wordProbs, key=wordProbs.get)

This next cell contains the code for the sentence prediction/spellchecking model. The following cell can be used to test it by inputting a sentence after running it.

In [ ]:
def sentence_prediction(sent: list, punct=punctuation, nns=nouns, prediction_length=5):
    output = []

    # Localize globals for speed
    close_words_fn = close_words
    best_word_fn = best_word
    nbest_fn = nbest_continuations
    model_params = (counts, prefix_denominator, total_counts)
    append = output.append
    stop = nns | punct

    # spellcheck first
    for i, word in enumerate(sent):
        # special cases: , . ? ! ; - --, etc.  (signals to end parsing)
        if word in punct:
            append(word)
            continue
            # TODO: make cases for (), which treats what's inside as its own sentence to be analzyed, but still remembers what's outside the ()

        closest_words = close_words_fn(word)
        correct_word = best_word_fn(sent, i, closest_words, *model_params)
        append(correct_word)

    # predict next word until next word is noun or punctuation
    for _ in range(prediction_length):
        print(output)
        nbest = nbest_fn(output, *model_params, 1)
        next_word = nbest[0][0] if not isinstance(nbest[0][0], tuple) else nbest[0][0][0]
        append(next_word)
        
        if next_word in stop: # maybe add: and i > 3 (replace _ with i)
            break

    return output

In [17]:
print("input a sentence:")
sent = input()

pattern = r"\d+[:.]\d+|\w+(?:'\w+)?|[.,!?;:()\-—]"
sent = re.findall(pattern, sent)

sentence_prediction(sent)

# predict next word
# nbest_continuations(sent, ...)

input a sentence:
['he']


['he', 'loss']